# Build `cusip_financial_data` (EODHD → Postgres)

Implements `Insert_Fundamental.md`:

1. Pull the required `(cusip, year, quarter)` universe from `stocks_return`.
2. Map cusip → ticker via `ticker_to_cusip` (append `.US` suffix).
3. Call EODHD fundamentals once per ticker and parse quarterly Income / Balance / Cash-Flow.
4. Compute node features (`diluted_eps`, `roe`, `debt_to_equity`, `fcf_per_share`, ...).
5. Insert matching rows into `cusip_financial_data`.

Destructive writes are gated by `DRY_RUN`. API key is read from `.env` (`EDOHD_API`).

## Setup

In [15]:
import os
import sys
import time
import warnings
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv
from psycopg2.extras import execute_batch

warnings.filterwarnings("ignore", category=UserWarning)

for env_path in [Path.cwd() / ".env", Path.cwd().parent / ".env", Path.cwd().parent.parent / ".env"]:
    if env_path.exists():
        load_dotenv(env_path)
        print("Loaded .env from:", env_path.resolve())
        break

def project_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, cwd.parent, cwd.parent.parent]:
        if (p / "ETL").is_dir():
            return p
    return cwd.parent.parent

ROOT = project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ETL.gnn_db_pipeline.config import (
    TARGET_DB,
    TGT_TICKER_TO_CUSIP,
    TGT_STOCKS_RETURN,
)
from ETL.gnn_db_pipeline.db_connector import ConfigurablePostgresHandler

TGT_CUSIP_FINANCIAL = "cusip_financial_data"

DRY_RUN = False   # flip to False to create table, call EODHD, and insert rows
MAX_TICKERS = None  # set to an int (e.g. 5) to cap the fetch loop while testing

API_KEY = os.getenv("EDOHD_API") or os.getenv("EODHD_API")
assert API_KEY, "EDOHD_API (or EODHD_API) missing from .env"

handler = ConfigurablePostgresHandler(TARGET_DB)
handler.connect()
print("Connected to", TARGET_DB, "| DRY_RUN =", DRY_RUN)

2026-04-25 15:39:57 - ETL_Pipeline - INFO - Connected to PostgreSQL: postgres@127.0.0.1:5432/13FGNN


Loaded .env from: C:\Users\potda\Daniel\BGU\Year_D\סמסטר ז\Final_Project\Social-Network-Stock-Market\.env
Connected to 13FGNN | DRY_RUN = False


## 1. Create target table `cusip_financial_data`

Primary key `(cusip, year, quarter)` so re-runs are idempotent via `ON CONFLICT DO NOTHING`.

In [16]:
create_target_ddl = f"""
CREATE TABLE IF NOT EXISTS {TGT_CUSIP_FINANCIAL} (
    cusip           TEXT     NOT NULL,
    year            SMALLINT NOT NULL,
    quarter         SMALLINT NOT NULL,
    diluted_eps     DOUBLE PRECISION,
    roe             DOUBLE PRECISION,
    ev_ebitda       DOUBLE PRECISION,
    pe_ratio        DOUBLE PRECISION,
    price_to_sales  DOUBLE PRECISION,
    price_to_book   DOUBLE PRECISION,
    debt_to_equity  DOUBLE PRECISION,
    dividend_yield  DOUBLE PRECISION,
    fcf_per_share   DOUBLE PRECISION,
    PRIMARY KEY (cusip, year, quarter)
);
"""

if DRY_RUN:
    print("[DRY_RUN] would execute:\n" + create_target_ddl)
else:
    handler.execute(create_target_ddl)
    print("Table ready:", TGT_CUSIP_FINANCIAL)

Table ready: cusip_financial_data


## 2. Load the `(cusip, year, quarter)` universe from `stocks_return`

In [17]:
universe = handler.query(f"""
    SELECT DISTINCT cusip, year, quarter
    FROM {TGT_STOCKS_RETURN}
""")
universe["year"] = universe["year"].astype(int)
universe["quarter"] = universe["quarter"].astype(int)
universe_keys = set(map(tuple, universe[["cusip", "year", "quarter"]].itertuples(index=False, name=None)))
print(f"Universe: {len(universe):,} (cusip, year, quarter) rows, "
      f"{universe['cusip'].nunique():,} distinct cusips")
universe.head()

Universe: 151,904 (cusip, year, quarter) rows, 4,463 distinct cusips


,cusip,year,quarter
0,278768106,2015,4
1,03837C106,2017,3
2,565788106,2016,2
3,770323103,2014,3
4,47805L101,2023,2


## 3. Map cusip → ticker

Only cusips that exist in the universe and have a ticker populated. Tickers are normalized to the `AAPL.US` format EODHD expects.

In [18]:
t2c = handler.query(f"""
    SELECT cusip, ticker
    FROM {TGT_TICKER_TO_CUSIP}
    WHERE ticker IS NOT NULL AND ticker <> ''
""")

mapping = t2c.merge(universe[["cusip"]].drop_duplicates(), on="cusip", how="inner")

def normalize_ticker(t: str) -> str:
    t = (t or "").strip().upper()
    if not t:
        return t
    return t if "." in t else f"{t}.US"

mapping["ticker_eod"] = mapping["ticker"].map(normalize_ticker)
mapping = mapping[mapping["ticker_eod"].str.len() > 0].drop_duplicates()

print(f"cusip/ticker pairs in universe: {len(mapping):,}")
print(f"distinct tickers to fetch:      {mapping['ticker_eod'].nunique():,}")
print(f"cusips in universe w/o ticker:  {universe['cusip'].nunique() - mapping['cusip'].nunique():,}")
mapping.head()

cusip/ticker pairs in universe: 4,463
distinct tickers to fetch:      4,463
cusips in universe w/o ticker:  0


,cusip,ticker,ticker_eod
0,808194104,SHLM,SHLM.US
1,831865209,AOS,AOS.US
2,000360206,AAON,AAON.US
3,000361105,AIR,AIR.US
4,003654100,ABMD,ABMD.US


## 4. EODHD fetch + parsing helpers

Follows the field mapping in `Insert_Fundamental.md`:

* Income: `netIncome`, `totalRevenue`, `ebitda`
* Balance: `totalStockholderEquity`, `commonStockSharesOutstanding`, `shortLongTermDebtTotal` (fallback `shortTermDebt + longTermDebt`)
* Cash-Flow: `freeCashFlow`

Derived: `roe = net_income / equity`, `debt_to_equity = debt / equity`, `eps = net_income / shares`, `fcf_per_share = fcf / shares`. Ratios that need price data (`ev_ebitda`, `pe_ratio`, `price_to_sales`, `price_to_book`, `dividend_yield`) are left `NULL` — fill them in a follow-up step from `ticker_prices`.

In [19]:
EODHD_URL = "https://eodhd.com/api/fundamentals/{ticker}?api_token={key}&fmt=json"


def fetch_json_safe(url, retries: int = 3, backoff: float = 1.5):
    for attempt in range(1, retries + 1):
        try:
            res = requests.get(url, timeout=30)
            if res.status_code == 404:
                return None
            res.raise_for_status()
            return res.json()
        except Exception as err:
            if attempt == retries:
                print(f"  ! request failed: {err}")
                return None
            time.sleep(backoff ** attempt)
    return None


def fetch_fundamentals(ticker: str):
    return fetch_json_safe(EODHD_URL.format(ticker=ticker, key=API_KEY))


def extract_yq(date_str):
    d = pd.to_datetime(date_str)
    return d.year, (d.month - 1) // 3 + 1


def safe_float(val):
    if val in (None, "", "0"):
        return 0.0
    try:
        return float(val)
    except Exception:
        return 0.0


def get_debt(row):
    primary = row.get("shortLongTermDebtTotal")
    if primary in (None, ""):
        return safe_float(row.get("shortTermDebt")) + safe_float(row.get("longTermDebt"))
    return safe_float(primary)


def get_cash(row):
    primary = row.get("cashAndShortTermInvestments")
    if primary not in (None, ""):
        return safe_float(primary)
    return safe_float(row.get("cash")) + safe_float(row.get("shortTermInvestments"))


def get_fcf(cf_row):
    fcf = safe_float(cf_row.get("freeCashFlow"))
    if fcf:
        return fcf
    op = safe_float(cf_row.get("totalCashFromOperatingActivities"))
    capex = safe_float(cf_row.get("capitalExpenditures"))
    if op:
        return op - capex
    return 0.0


def parse_quarters(data):
    """Per-quarter feature dicts. Underscore-prefixed keys are raw values
    used for TTM aggregation / market-data ratios downstream and dropped before insert."""
    if not data:
        return []
    fin = data.get("Financials", {}) or {}
    inc = (fin.get("Income_Statement", {}) or {}).get("quarterly", {}) or {}
    bs = (fin.get("Balance_Sheet", {}) or {}).get("quarterly", {}) or {}
    cf = (fin.get("Cash_Flow", {}) or {}).get("quarterly", {}) or {}

    rows = []
    for date in inc.keys():
        try:
            year, quarter = extract_yq(date)
        except Exception:
            continue

        inc_row = inc.get(date, {}) or {}
        bs_row = bs.get(date, {}) or {}
        cf_row = cf.get(date, {}) or {}

        net_income = safe_float(inc_row.get("netIncome"))
        revenue = safe_float(inc_row.get("totalRevenue"))
        ebitda = safe_float(inc_row.get("ebitda"))
        equity = safe_float(bs_row.get("totalStockholderEquity"))
        shares = safe_float(bs_row.get("commonStockSharesOutstanding"))
        debt = get_debt(bs_row)
        cash = get_cash(bs_row)
        fcf = get_fcf(cf_row)

        rows.append({
            "year": int(year),
            "quarter": int(quarter),
            "_report_date": date,
            "_net_income": net_income,
            "_revenue": revenue,
            "_ebitda": ebitda,
            "_equity": equity,
            "_shares": shares,
            "_debt": debt,
            "_cash": cash,
            "diluted_eps": (net_income / shares) if shares else None,
            "roe": (net_income / equity) if equity else None,
            "ev_ebitda": None,
            "pe_ratio": None,
            "price_to_sales": None,
            "price_to_book": None,
            "debt_to_equity": (debt / equity) if equity else None,
            "dividend_yield": None,
            "fcf_per_share": (fcf / shares) if shares else None,
        })
    return rows

## 5. Fetch fundamentals once per ticker

Cache the parsed payload by ticker — if the same ticker is mapped to several cusips we still only burn one EODHD call. Flip `DRY_RUN = False` (and optionally set `MAX_TICKERS = 5` first) before running; this burns API credits.

In [20]:
all_rows = []
skipped = []
ticker_cache = {}

pairs = mapping[["cusip", "ticker_eod"]].drop_duplicates().reset_index(drop=True)
if MAX_TICKERS is not None:
    capped_tickers = pairs["ticker_eod"].drop_duplicates().head(MAX_TICKERS)
    pairs = pairs[pairs["ticker_eod"].isin(capped_tickers)].reset_index(drop=True)

if DRY_RUN:
    print(f"[DRY_RUN] would fetch {pairs['ticker_eod'].nunique():,} tickers "
          f"across {len(pairs):,} (cusip, ticker) pairs.")
else:
    total = len(pairs)
    for i, (cusip, ticker) in enumerate(pairs.itertuples(index=False), start=1):
        if ticker not in ticker_cache:
            data = fetch_fundamentals(ticker)
            ticker_cache[ticker] = parse_quarters(data)

        quarters = ticker_cache[ticker]
        if not quarters:
            skipped.append((cusip, ticker))
            continue

        for q in quarters:
            all_rows.append({"cusip": cusip, "ticker": ticker, **q})

        if i % 50 == 0 or i == total:
            print(f"  {i}/{total} pairs processed "
                  f"(rows={len(all_rows):,}, skipped={len(skipped):,})")

    print(f"\nCollected {len(all_rows):,} quarterly rows; "
          f"{len(skipped):,} (cusip, ticker) pairs had no data.")

  50/4463 pairs processed (rows=5,033, skipped=0)
  100/4463 pairs processed (rows=10,281, skipped=0)
  150/4463 pairs processed (rows=15,337, skipped=1)
  200/4463 pairs processed (rows=20,522, skipped=2)
  250/4463 pairs processed (rows=25,850, skipped=3)
  300/4463 pairs processed (rows=31,558, skipped=4)
  350/4463 pairs processed (rows=36,179, skipped=4)
  400/4463 pairs processed (rows=41,186, skipped=7)
  450/4463 pairs processed (rows=46,290, skipped=9)
  500/4463 pairs processed (rows=51,534, skipped=9)
  550/4463 pairs processed (rows=57,600, skipped=9)
  600/4463 pairs processed (rows=63,182, skipped=9)
  650/4463 pairs processed (rows=68,688, skipped=10)
  700/4463 pairs processed (rows=73,879, skipped=11)
  750/4463 pairs processed (rows=78,853, skipped=13)
  800/4463 pairs processed (rows=82,977, skipped=15)


KeyboardInterrupt: 

In [21]:
from concurrent.futures import ThreadPoolExecutor, as_completed

all_tickers = pairs["ticker_eod"].drop_duplicates().tolist()
remaining = [t for t in all_tickers if t not in ticker_cache]
print(f"already fetched: {len(ticker_cache):,}  remaining: {len(remaining):,}")

def fetch_one(ticker):
    return ticker, parse_quarters(fetch_fundamentals(ticker))

MAX_WORKERS = 20  # EODHD tolerates this fine; bump to 30 if rate allows
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futs = {ex.submit(fetch_one, t): t for t in remaining}
    done = 0
    for f in as_completed(futs):
        t, qs = f.result()
        ticker_cache[t] = qs
        done += 1
        if done % 100 == 0 or done == len(remaining):
            print(f"  {done}/{len(remaining)}  (cache size: {len(ticker_cache):,})")

# Rebuild all_rows from the full cache
all_rows = []
skipped = []
for cusip, ticker in pairs.itertuples(index=False):
    qs = ticker_cache.get(ticker, [])
    if not qs:
        skipped.append((cusip, ticker))
        continue
    for q in qs:
        all_rows.append({"cusip": cusip, "ticker": ticker, **q})

print(f"\nCollected {len(all_rows):,} quarterly rows; {len(skipped):,} skipped.")

already fetched: 812  remaining: 3,651
  100/3651  (cache size: 912)
  200/3651  (cache size: 1,012)
  300/3651  (cache size: 1,112)
  400/3651  (cache size: 1,212)
  500/3651  (cache size: 1,312)
  600/3651  (cache size: 1,412)
  700/3651  (cache size: 1,512)
  800/3651  (cache size: 1,612)
  900/3651  (cache size: 1,712)
  1000/3651  (cache size: 1,812)
  1100/3651  (cache size: 1,912)
  1200/3651  (cache size: 2,012)
  1300/3651  (cache size: 2,112)
  1400/3651  (cache size: 2,212)
  1500/3651  (cache size: 2,312)
  1600/3651  (cache size: 2,412)
  1700/3651  (cache size: 2,512)
  1800/3651  (cache size: 2,612)
  1900/3651  (cache size: 2,712)
  2000/3651  (cache size: 2,812)
  2100/3651  (cache size: 2,912)
  2200/3651  (cache size: 3,012)
  2300/3651  (cache size: 3,112)
  2400/3651  (cache size: 3,212)
  2500/3651  (cache size: 3,312)
  2600/3651  (cache size: 3,412)
  2700/3651  (cache size: 3,512)
  2800/3651  (cache size: 3,612)
  2900/3651  (cache size: 3,712)
  3000/3651  (c

## 5b. Enrich with prices + dividends (yfinance)

Two `yfinance` calls per ticker (no API key needed):

* `Ticker.history(...)` — `Adj Close` for the last trading day ≤ quarter-end.
* `Dividends` column from the same history call — sum over the trailing 365 days.

TTM aggregates (revenue, ebitda, net income) = rolling 4 reported quarters per cusip.

```
market_cap        = price * shares
pe_ratio          = price / (net_income_TTM / shares)
price_to_sales    = market_cap / revenue_TTM
price_to_book     = market_cap / equity (MRQ)
ev_ebitda         = (market_cap + debt - cash) / ebitda_TTM
dividend_yield    = dividends_TTM / price
```

In [22]:
from concurrent.futures import ThreadPoolExecutor, as_completed

EOD_URL = "https://eodhd.com/api/eod/{ticker}?api_token={key}&fmt=json&from={start}&to={end}"
DIV_URL = "https://eodhd.com/api/div/{ticker}?api_token={key}&fmt=json&from={start}&to={end}"

QUARTER_END_MD = {1: (3, 31), 2: (6, 30), 3: (9, 30), 4: (12, 31)}

def quarter_end_ts(year, quarter):
    m, d = QUARTER_END_MD[int(quarter)]
    return pd.Timestamp(int(year), m, d)


def fetch_eod_series(ticker, start, end):
    data = fetch_json_safe(EOD_URL.format(ticker=ticker, key=API_KEY, start=start, end=end))
    if not data:
        return pd.Series(dtype=float)
    df = pd.DataFrame(data)
    if df.empty or "date" not in df.columns:
        return pd.Series(dtype=float)
    price_col = "adjusted_close" if "adjusted_close" in df.columns else "close"
    df["date"] = pd.to_datetime(df["date"])
    return df.sort_values("date").set_index("date")[price_col].astype(float)


def fetch_div_series(ticker, start, end):
    data = fetch_json_safe(DIV_URL.format(ticker=ticker, key=API_KEY, start=start, end=end))
    if not data:
        return pd.Series(dtype=float)
    df = pd.DataFrame(data)
    if df.empty:
        return pd.Series(dtype=float)
    date_col = next((c for c in ("date", "exDate") if c in df.columns), None)
    if not date_col or "value" not in df.columns:
        return pd.Series(dtype=float)
    df["d"] = pd.to_datetime(df[date_col])
    return df.sort_values("d").set_index("d")["value"].astype(float)


def safe_div(num, den):
    if num is None or den is None:
        return None
    try:
        if pd.isna(num) or pd.isna(den) or den == 0:
            return None
        return float(num) / float(den)
    except Exception:
        return None


  # Resumable caches: persist across reruns/interrupts so we don't re-fetch.
try:
    eod_cache
except NameError:
    eod_cache = {}
try:
    div_cache
except NameError:
    div_cache = {}


if DRY_RUN or not all_rows:
    print("[DRY_RUN or no rows] skipping enrichment.")
    enriched_df = pd.DataFrame()
else:
    base = (pd.DataFrame(all_rows)
              .drop_duplicates(subset=["cusip", "year", "quarter"])
              .sort_values(["cusip", "year", "quarter"])
              .reset_index(drop=True))

    for out, raw in [("_net_income_ttm", "_net_income"),
                     ("_revenue_ttm", "_revenue"),
                     ("_ebitda_ttm", "_ebitda")]:
        base[out] = (base.groupby("cusip")[raw]
                         .transform(lambda s: s.rolling(4, min_periods=4).sum()))

    base["_q_end"] = [quarter_end_ts(y, q) for y, q in zip(base["year"], base["quarter"])]

    # Per-ticker date range (covers all its quarter-ends in base)
    ranges = (base.groupby("ticker")["_q_end"]
                  .agg(["min", "max"])
                  .rename(columns={"min": "qmin", "max": "qmax"}))
    ranges["start"] = (ranges["qmin"] - pd.Timedelta(days=400)).dt.date.astype(str)
    ranges["end"] = (ranges["qmax"] + pd.Timedelta(days=10)).dt.date.astype(str)

    tickers = ranges.index.tolist()
    remaining = [t for t in tickers if t not in eod_cache or t not in div_cache]
    print(f"fetching EOD + dividends: {len(tickers)} total, "
          f"{len(tickers) - len(remaining):,} cached, {len(remaining):,} remaining...")

    def fetch_pair(ticker):
        s = ranges.loc[ticker, "start"]
        e = ranges.loc[ticker, "end"]
        return ticker, fetch_eod_series(ticker, s, e), fetch_div_series(ticker, s, e)

    MAX_WORKERS = 20
    if remaining:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futs = {ex.submit(fetch_pair, t): t for t in remaining}
            done = 0
            for f in as_completed(futs):
                t, prices, divs = f.result()
                eod_cache[t] = prices
                div_cache[t] = divs
                done += 1
                if done % 100 == 0 or done == len(remaining):
                    print(f"  {done}/{len(remaining)}")

    no_data = [t for t in tickers if eod_cache.get(t) is None or eod_cache[t].empty]
    if no_data:
        print(f"\nno price data for {len(no_data)} tickers, e.g.: {no_data[:10]}")

    def price_at(ticker, ts):
        s = eod_cache.get(ticker)
        if s is None or s.empty:
            return None
        s = s[s.index <= ts]
        return float(s.iloc[-1]) if not s.empty else None

    def div_ttm(ticker, ts):
        s = div_cache.get(ticker)
        if s is None or s.empty:
            return 0.0
        window = s[(s.index > ts - pd.Timedelta(days=365)) & (s.index <= ts)]
        return float(window.sum())

    base["_price"] = [price_at(t, d) for t, d in zip(base["ticker"], base["_q_end"])]
    base["_div_ttm"] = [div_ttm(t, d) for t, d in zip(base["ticker"], base["_q_end"])]
    base["_market_cap"] = base.apply(
        lambda r: r["_price"] * r["_shares"]
        if (r["_price"] is not None and r["_shares"]) else None,
        axis=1,
    )

    base["pe_ratio"] = base.apply(
        lambda r: safe_div(r["_price"],
                           safe_div(r["_net_income_ttm"], r["_shares"])),
        axis=1,
    )
    base["price_to_sales"] = base.apply(
        lambda r: safe_div(r["_market_cap"], r["_revenue_ttm"]), axis=1)
    base["price_to_book"] = base.apply(
        lambda r: safe_div(r["_market_cap"], r["_equity"]), axis=1)
    base["ev_ebitda"] = base.apply(
        lambda r: safe_div(
            (r["_market_cap"] + (r["_debt"] or 0) - (r["_cash"] or 0))
            if r["_market_cap"] is not None else None,
            r["_ebitda_ttm"],
        ),
        axis=1,
    )
    base["dividend_yield"] = base.apply(
        lambda r: safe_div(r["_div_ttm"], r["_price"]), axis=1)

    enriched_df = base
    print(f"\nEnriched: {len(enriched_df):,} rows")
    display(enriched_df[[
        "cusip", "year", "quarter",
        "pe_ratio", "price_to_sales", "price_to_book",
        "ev_ebitda", "dividend_yield",
    ]].head())

fetching EOD + dividends: 4384 total, 5 cached, 4,379 remaining...
  100/4379
  200/4379
  300/4379
  400/4379
  500/4379
  600/4379
  700/4379
  800/4379
  900/4379
  1000/4379
  1100/4379
  1200/4379
  1300/4379
  1400/4379
  1500/4379
  1600/4379
  1700/4379
  1800/4379
  1900/4379
  2000/4379
  2100/4379
  2200/4379
  2300/4379
  2400/4379
  2500/4379
  2600/4379
  2700/4379
  2800/4379
  2900/4379
  3000/4379
  3100/4379
  3200/4379
  3300/4379
  3400/4379
  3500/4379
  3600/4379
  3700/4379
  3800/4379
  3900/4379
  4000/4379
  4100/4379
  4200/4379
  4300/4379
  4379/4379

no price data for 9 tickers, e.g.: ['ALNAQ.US', 'APC.US', 'EMGC.US', 'NRTSF.US', 'PHH.US', 'RNA.US', 'ROC.US', 'SUNEQ.US', 'TLGTQ.US']

Enriched: 356,986 rows


,cusip,year,quarter,pe_ratio,price_to_sales,price_to_book,ev_ebitda,dividend_yield
0,000307108,2014,3,NaN,NaN,NaN,NaN,NaN
1,000307108,2016,1,NaN,NaN,3.110774,NaN,0.0
2,000307108,2016,2,NaN,NaN,3.305790,NaN,0.0
3,000307108,2016,3,296.371282,1.689442,2.520805,NaN,0.0
4,000307108,2016,4,NaN,NaN,NaN,NaN,0.0


In [ ]:
EOD_URL = "https://eodhd.com/api/eod/{ticker}?api_token={key}&fmt=json&from={start}&to={end}"
DIV_URL = "https://eodhd.com/api/div/{ticker}?api_token={key}&fmt=json&from={start}&to={end}"

QUARTER_END_MD = {1: (3, 31), 2: (6, 30), 3: (9, 30), 4: (12, 31)}


def quarter_end_ts(year, quarter):
    m, d = QUARTER_END_MD[int(quarter)]
    return pd.Timestamp(int(year), m, d)


def fetch_eod_series(ticker, start, end):
    data = fetch_json_safe(EOD_URL.format(ticker=ticker, key=API_KEY, start=start, end=end))
    if not data:
        return pd.Series(dtype=float)
    df = pd.DataFrame(data)
    if df.empty or "date" not in df.columns:
        return pd.Series(dtype=float)
    price_col = "adjusted_close" if "adjusted_close" in df.columns else "close"
    df["date"] = pd.to_datetime(df["date"])
    return df.sort_values("date").set_index("date")[price_col].astype(float)


def fetch_div_series(ticker, start, end):
    data = fetch_json_safe(DIV_URL.format(ticker=ticker, key=API_KEY, start=start, end=end))
    if not data:
        return pd.Series(dtype=float)
    df = pd.DataFrame(data)
    if df.empty:
        return pd.Series(dtype=float)
    date_col = next((c for c in ("date", "exDate") if c in df.columns), None)
    if not date_col or "value" not in df.columns:
        return pd.Series(dtype=float)
    df["d"] = pd.to_datetime(df[date_col])
    return df.sort_values("d").set_index("d")["value"].astype(float)


def safe_div(num, den):
    if num is None or den is None:
        return None
    try:
        if pd.isna(num) or pd.isna(den) or den == 0:
            return None
        return float(num) / float(den)
    except Exception:
        return None


if DRY_RUN or not all_rows:
    print("[DRY_RUN or no rows] skipping enrichment.")
    enriched_df = pd.DataFrame()
else:
    base = (pd.DataFrame(all_rows)
              .drop_duplicates(subset=["cusip", "year", "quarter"])
              .sort_values(["cusip", "year", "quarter"])
              .reset_index(drop=True))

    for out, raw in [("_net_income_ttm", "_net_income"),
                     ("_revenue_ttm", "_revenue"),
                     ("_ebitda_ttm", "_ebitda")]:
        base[out] = (base.groupby("cusip")[raw]
                         .transform(lambda s: s.rolling(4, min_periods=4).sum()))

    base["_q_end"] = [quarter_end_ts(y, q) for y, q in zip(base["year"], base["quarter"])]

    eod_cache, div_cache = {}, {}
    no_data = []
    tickers = base["ticker"].dropna().unique().tolist()
    print(f"fetching EOD + dividends for {len(tickers)} tickers...")

    for i, t in enumerate(tickers, 1):
        sub = base[base["ticker"] == t]
        start = (sub["_q_end"].min() - pd.Timedelta(days=400)).date().isoformat()
        end = (sub["_q_end"].max() + pd.Timedelta(days=10)).date().isoformat()
        eod_cache[t] = fetch_eod_series(t, start, end)
        div_cache[t] = fetch_div_series(t, start, end)
        if eod_cache[t].empty:
            no_data.append(t)
        if i % 25 == 0 or i == len(tickers):
            print(f"  {i}/{len(tickers)}  (empty so far: {len(no_data)})")

    if no_data:
        print(f"\nno price data for {len(no_data)} tickers, e.g.: {no_data[:10]}")

    def price_at(ticker, ts):
        s = eod_cache.get(ticker)
        if s is None or s.empty:
            return None
        s = s[s.index <= ts]
        return float(s.iloc[-1]) if not s.empty else None

    def div_ttm(ticker, ts):
        s = div_cache.get(ticker)
        if s is None or s.empty:
            return 0.0
        window = s[(s.index > ts - pd.Timedelta(days=365)) & (s.index <= ts)]
        return float(window.sum())

    base["_price"] = [price_at(t, d) for t, d in zip(base["ticker"], base["_q_end"])]
    base["_div_ttm"] = [div_ttm(t, d) for t, d in zip(base["ticker"], base["_q_end"])]
    base["_market_cap"] = base.apply(
        lambda r: r["_price"] * r["_shares"]
        if (r["_price"] is not None and r["_shares"]) else None,
        axis=1,
    )

    base["pe_ratio"] = base.apply(
        lambda r: safe_div(r["_price"],
                           safe_div(r["_net_income_ttm"], r["_shares"])),
        axis=1,
    )
    base["price_to_sales"] = base.apply(
        lambda r: safe_div(r["_market_cap"], r["_revenue_ttm"]), axis=1)
    base["price_to_book"] = base.apply(
        lambda r: safe_div(r["_market_cap"], r["_equity"]), axis=1)
    base["ev_ebitda"] = base.apply(
        lambda r: safe_div(
            (r["_market_cap"] + (r["_debt"] or 0) - (r["_cash"] or 0))
            if r["_market_cap"] is not None else None,
            r["_ebitda_ttm"],
        ),
        axis=1,
    )
    base["dividend_yield"] = base.apply(
        lambda r: safe_div(r["_div_ttm"], r["_price"]), axis=1)

    enriched_df = base
    print(f"\nEnriched: {len(enriched_df):,} rows")
    display(enriched_df[[
        "cusip", "year", "quarter",
        "pe_ratio", "price_to_sales", "price_to_book",
        "ev_ebitda", "dividend_yield",
    ]].head())

fetching EOD + dividends for 5 tickers...
  5/5  (empty so far: 0)

Enriched: 689 rows


,cusip,year,quarter,pe_ratio,price_to_sales,price_to_book,ev_ebitda,dividend_yield
0,000360206,1990,1,NaN,NaN,NaN,NaN,NaN
1,000360206,1990,2,NaN,NaN,NaN,NaN,NaN
2,000360206,1990,3,NaN,NaN,NaN,NaN,NaN
3,000360206,1990,4,NaN,NaN,NaN,NaN,NaN
4,000360206,1991,1,3.661105,0.087429,1.830553,5.376765,0.0


## 6. Filter rows to the `stocks_return` universe

Only keep `(cusip, year, quarter)` triples that actually appear in `stocks_return` — that's the whole point of the table.

In [23]:
FINAL_COLS = [
    "cusip", "year", "quarter",
    "diluted_eps", "roe", "ev_ebitda",
    "pe_ratio", "price_to_sales", "price_to_book",
    "debt_to_equity", "dividend_yield", "fcf_per_share",
]

if DRY_RUN or enriched_df.empty:
    print("[DRY_RUN or no rows] skipping filter step.")
    financials_df = pd.DataFrame()
else:
    keep_mask = [
        (c, y, q) in universe_keys
        for c, y, q in enriched_df[["cusip", "year", "quarter"]].itertuples(index=False, name=None)
    ]
    financials_df = enriched_df[keep_mask][FINAL_COLS].reset_index(drop=True)
    print(f"Rows matching universe: {len(financials_df):,}")
    display(financials_df.head())

Rows matching universe: 147,283


,cusip,year,quarter,diluted_eps,roe,ev_ebitda,pe_ratio,price_to_sales,price_to_book,debt_to_equity,dividend_yield,fcf_per_share
0,000307108,2016,1,0.025686,0.004038,NaN,NaN,NaN,3.110774,0.985889,0.0,-0.120015
1,000307108,2016,2,0.036799,0.005331,NaN,NaN,NaN,3.305790,1.024129,0.0,-0.451085
2,000307108,2016,3,-0.106512,-0.015440,NaN,296.371282,1.689442,2.520805,1.158285,0.0,-0.642952
3,000307108,2016,4,NaN,0.002895,NaN,NaN,NaN,NaN,1.145361,0.0,NaN
4,000307108,2017,1,NaN,-0.003614,NaN,NaN,NaN,NaN,1.189125,0.0,NaN


In [24]:
print("DataFrame shape:", financials_df.shape)
print("\nNon-null (not None and not NaN) count per column:")
print(financials_df.count())

print("\nNull (None or NaN) count per column:")
print(financials_df.isnull().sum())

DataFrame shape: (147283, 12)

Non-null (not None and not NaN) count per column:
cusip             147283
year              147283
quarter           147283
diluted_eps       142367
roe               146878
ev_ebitda         136557
pe_ratio          142076
price_to_sales    138257
price_to_book     142108
debt_to_equity    146878
dividend_yield    147262
fcf_per_share     142367
dtype: int64

Null (None or NaN) count per column:
cusip                 0
year                  0
quarter               0
diluted_eps        4916
roe                 405
ev_ebitda         10726
pe_ratio           5207
price_to_sales     9026
price_to_book      5175
debt_to_equity      405
dividend_yield       21
fcf_per_share      4916
dtype: int64


In [ ]:
financials_df.describe()

,year,quarter,diluted_eps,roe,ev_ebitda,pe_ratio,price_to_sales,price_to_book,debt_to_equity,dividend_yield,fcf_per_share
count,201.000000,201.000000,181.000000,201.000000,181.000000,181.000000,181.000000,181.000000,201.000000,201.000000,181.000000
mean,2018.398010,2.502488,0.457263,0.027910,23.241294,41.310660,4.783494,5.747011,0.438677,0.009592,0.362888
std,3.397175,1.118590,0.473959,0.176006,39.532115,69.925056,5.450649,4.268886,1.070853,0.008518,0.651805
min,2013.000000,1.000000,-1.071633,-2.406017,-381.907900,-447.453125,0.281028,0.670730,0.000000,0.000000,-2.242165
25%,2016.000000,2.000000,0.200000,0.020418,11.953990,20.588065,1.013174,1.922671,0.039580,0.000000,0.073771
50%,2018.000000,3.000000,0.395770,0.040128,16.146691,28.619541,2.810265,5.146646,0.170784,0.008118,0.299262
75%,2021.000000,3.000000,0.648697,0.059560,28.679751,50.910361,5.142300,7.398896,0.269944,0.014514,0.798834
max,2025.000000,4.000000,2.321826,0.339121,169.616526,392.487375,30.072339,26.177195,6.680124,0.037390,1.829424


## 7. Insert into `cusip_financial_data`

`ON CONFLICT (cusip, year, quarter) DO NOTHING` — reruns are safe, partial fills won't be overwritten. Uses `execute_batch` for throughput.

In [25]:
INSERT_SQL = f"""
INSERT INTO {TGT_CUSIP_FINANCIAL} (
    cusip, year, quarter,
    diluted_eps, roe, ev_ebitda,
    pe_ratio, price_to_sales, price_to_book,
    debt_to_equity, dividend_yield, fcf_per_share
) VALUES (
    %(cusip)s, %(year)s, %(quarter)s,
    %(diluted_eps)s, %(roe)s, %(ev_ebitda)s,
    %(pe_ratio)s, %(price_to_sales)s, %(price_to_book)s,
    %(debt_to_equity)s, %(dividend_yield)s, %(fcf_per_share)s
)
ON CONFLICT (cusip, year, quarter) DO NOTHING;
"""

def insert_rows(records, page_size: int = 1000):
    if not records:
        print("no rows to insert.")
        return 0
    conn = handler.connection
    with conn.cursor() as cur:
        execute_batch(cur, INSERT_SQL, records, page_size=page_size)
    conn.commit()
    return len(records)


if DRY_RUN:
    print("[DRY_RUN] would insert rows with:\n" + INSERT_SQL)
elif financials_df.empty:
    print("no rows collected — nothing to insert.")
else:
    records = financials_df.where(pd.notna(financials_df), None).to_dict(orient="records")
    n = insert_rows(records)
    print(f"Inserted {n:,} rows (conflicts ignored).")

Inserted 147,283 rows (conflicts ignored).


## 8. Verify

Row count, coverage against the universe, and a sample.

In [26]:
if DRY_RUN:
    print("[DRY_RUN] skipping verification queries.")
else:
    stats = handler.query(f"""
        SELECT
            COUNT(*)                         AS total_rows,
            COUNT(DISTINCT cusip)            AS distinct_cusips,
            MIN(year) || 'Q' || MIN(quarter) AS earliest,
            MAX(year) || 'Q' || MAX(quarter) AS latest
        FROM {TGT_CUSIP_FINANCIAL}
    """)
    print(stats.T)

    coverage = handler.query(f"""
        WITH u AS (
            SELECT DISTINCT cusip, year, quarter FROM {TGT_STOCKS_RETURN}
        )
        SELECT
            (SELECT COUNT(*) FROM u)                     AS universe_rows,
            (SELECT COUNT(*) FROM {TGT_CUSIP_FINANCIAL}) AS financial_rows,
            (SELECT COUNT(*) FROM u
               JOIN {TGT_CUSIP_FINANCIAL} f USING (cusip, year, quarter)) AS covered,
            (SELECT COUNT(*) FROM u
               LEFT JOIN {TGT_CUSIP_FINANCIAL} f USING (cusip, year, quarter)
               WHERE f.cusip IS NULL)                    AS missing
    """)
    print(coverage.T)

    sample = handler.query(f"""
        SELECT * FROM {TGT_CUSIP_FINANCIAL}
        ORDER BY cusip, year, quarter
        LIMIT 10
    """)
    display(sample)

                      0
total_rows       147283
distinct_cusips    4365
earliest         2013Q1
latest           2025Q4
                     0
universe_rows   151904
financial_rows  147283
covered         147283
missing           4621


,cusip,year,quarter,diluted_eps,roe,ev_ebitda,pe_ratio,price_to_sales,price_to_book,debt_to_equity,dividend_yield,fcf_per_share
0,000307108,2016,1,0.025686,0.004038,NaN,NaN,NaN,3.110774,0.985889,0.0,-0.120015
1,000307108,2016,2,0.036799,0.005331,NaN,NaN,NaN,3.305790,1.024129,0.0,-0.451085
2,000307108,2016,3,-0.106512,-0.015440,NaN,296.371282,1.689442,2.520805,1.158285,0.0,-0.642952
3,000307108,2016,4,NaN,0.002895,NaN,NaN,NaN,NaN,1.145361,0.0,NaN
4,000307108,2017,1,NaN,-0.003614,NaN,NaN,NaN,NaN,1.189125,0.0,NaN
5,000307108,2017,2,NaN,-0.011473,NaN,NaN,NaN,NaN,1.275277,0.0,NaN
6,000307108,2017,3,NaN,0.004487,NaN,NaN,NaN,NaN,1.336698,0.0,NaN
7,000307108,2017,4,NaN,-0.124654,NaN,NaN,NaN,NaN,1.494854,0.0,NaN
8,000307108,2018,1,NaN,-0.001285,NaN,NaN,NaN,NaN,2.088250,0.0,NaN
9,000307108,2018,2,NaN,-0.019375,NaN,NaN,NaN,NaN,2.099810,0.0,NaN


## 9. Cleanup

In [27]:
handler.disconnect()

2026-04-25 16:42:01 - ETL_Pipeline - INFO - Disconnected from PostgreSQL
